In [15]:
import html, json
import pandas as pd

from IPython.display import HTML, display


# sample dataset
data = "data/gsv3/grooveseeker_radio/gsv3_radio_entities_altcountry_sample.json"

# full dataset - buy access - verdantintel.com
data = "../verdant_intelligence/exports/grooveseeker_entities/grooveseeker_radio_entities_altcountry.json"

In [16]:
song_df = pd.read_json(data)
song_df = song_df.sample(frac=1).reset_index(drop=True)

song_df.head(10)

,entity,song,spotify_url,popularity,album,image
0,Ian Flanigan,Through The Darkness,https://open.spotify.com/track/6CwA044e1D2EGje...,30,Through The Darkness,https://i.scdn.co/image/ab67616d0000b2738d7e48...
1,Moin,Qareeb,https://open.spotify.com/track/35hwpi55PUyJFXj...,47,Qareeb,https://i.scdn.co/image/ab67616d0000b273b937d8...
2,Olivia Harms,Just Like Yesterday,https://open.spotify.com/track/6MWtbxlNmBtTIgA...,6,Rhinestone Cowgirl,https://i.scdn.co/image/ab67616d0000b273443a58...
3,Jodi Jones,Unbreak My Name,https://open.spotify.com/track/5uCStABx0lH4U7v...,0,Unbreak My Name,https://i.scdn.co/image/ab67616d0000b273ebd820...
4,Crystal Bowersox,Me And Bobby McGee,https://open.spotify.com/track/6Ay405ywjQu9Pg5...,19,American Idol: Season 9,https://i.scdn.co/image/ab67616d0000b2730aab2e...
5,Kelly Willis,Whatever Way The Wind Blows,https://open.spotify.com/track/4VRYSHA3WzRQTYN...,25,Kelly Willis,https://i.scdn.co/image/ab67616d0000b273c4a3d6...
6,Noeline Hofmann,The Bullfighter,https://open.spotify.com/track/1HgKUjJRScAcZD5...,43,The Bullfighter,https://i.scdn.co/image/ab67616d0000b273626c43...
7,Brendan Walter,Flipside of Free,https://open.spotify.com/track/33Sy1AXkrnkxhU6...,19,I don't know what I'm doing yet,https://i.scdn.co/image/ab67616d0000b273b971fb...
8,Billy and the Dreamboats,Cotton Candy,https://open.spotify.com/track/0f2ATYMhojHTcEf...,3,How Did You Get Here?,https://i.scdn.co/image/ab67616d0000b273b93669...
9,Pixel Grip,Stamina,https://open.spotify.com/track/4RHxfvaaVBPti5C...,48,Percepticide: The Death of Reality,https://i.scdn.co/image/ab67616d0000b2735562d0...


In [17]:
song_df.shape

(1731, 6)

I have given you a sample dataset of 100 songs to explore. 

# Play Music

In [18]:
def show_radio(song_df):
    tracks = []

    for _, row in song_df.iterrows():
        tracks.append(
            {
                "song": row["song"],
                "artist": row["entity"],
                "url": row["spotify_url"],
            }
        )

    tracks_json = json.dumps(tracks)

    radio_html = """
    <!DOCTYPE html>
    <html>
    <body>

    <div id="radio-label" style="margin-bottom:10px;"></div>
    <div id="spotify-player"></div>

    <div style="margin-top:10px;">
        <button id="back">← Back</button>
        <button id="forward">Forward →</button>
    </div>

    <script src="https://open.spotify.com/embed/iframe-api/v1" async></script>

    <script>
    const tracks = %s;

    let position = 0;
    let controller = null;
    let advancing = false;

    function updateLabel() {
        const row = tracks[position];

        document.getElementById("radio-label").innerHTML =
            "<b>" + row.song + "</b><br>" +
            row.artist + "<br>" +
            "Song " + (position + 1) + " of " + tracks.length;
    }

    function loadSong(step, autoplay=false) {
        position = (position + step + tracks.length) %% tracks.length;

        updateLabel();
        advancing = false;

        controller.loadEntity(tracks[position].url);

        if (autoplay) {
            setTimeout(() => controller.play(), 500);
        }
    }

    window.onSpotifyIframeApiReady = (IFrameAPI) => {
        const element = document.getElementById("spotify-player");

        IFrameAPI.createController(
            element,
            {
                url: tracks[position].url,
                width: "100%%",
                height: 152
            },
            (EmbedController) => {
                controller = EmbedController;

                controller.addListener("playback_update", (event) => {
                    const state = event.data;

                    if (
                        state.duration > 0 &&
                        state.position >= state.duration - 750 &&
                        !advancing
                    ) {
                        advancing = true;
                        loadSong(1, true);
                    }
                });
            }
        );
    };

    document.getElementById("back").onclick = () => loadSong(-1);
    document.getElementById("forward").onclick = () => loadSong(1);

    updateLabel();
    </script>

    </body>
    </html>
    """ % tracks_json

    display(
        HTML(
            '<iframe srcdoc="{}" width="100%" height="260" '
            'style="border:0;"></iframe>'.format(
                html.escape(radio_html, quote=True)
            )
        )
    )

In [20]:
song_df = song_df.sample(frac=1).reset_index(drop=True)

show_radio(song_df)

In [ ]:
'''
Stavenger - Brady Tucson
Little Boxes - Katie Pruitt
Bench Seat - Low Gap


'''